# Three-plane super-resolution T1 mapping — BART LLR + ANTs + PSIR

Adapts `T1map_US4_LLR_V11.ipynb` (single-orientation IR T1 mapping) to the
multi-orientation super-resolution approach of Deoni et al., MRM 2022;88:1273
(`MRM-88-1273.pdf`, reference code `MRM_29273_*.py`).

**Acquisition.** Four inversion times (TI) in each of three orientations (axial, coronal,
sagittal): 12 undersampled 3D IR-FSE scans at 1.8 x 1.8 x 5 mm, plus a fully sampled
`T1_graywhite_axi` scan used only for coil sensitivities. Target: 1.8 mm isotropic T1.

**Pipeline** (built up step by step; each step ends at a checkpoint):

| step | what | status |
|---|---|---|
| 1 | per-plane joint LLR reconstruction of the 4 TIs, self-calibrated coil maps, lambda/iteration sweep | **this version** |
| 2 | even/odd echo registration, PSIR, per-plane (anisotropic) T1 maps | |
| 3 | all planes in the common scanner frame | partly here (geometry) |
| 4 | inter-plane registration and isotropic template | |
| 5 | per-TI combination in isotropic space, T1 fit, export | |
| 6 | refinements: model-based SR, denoising, effective-TR correction | |

**Differences from the paper.** The paper acquired one TE per orientation. Here every
orientation has all four TIs, so each TI can be super-resolved from three planes before
fitting, and each plane also yields its own T1 map as a voxel-wise reference.

**Changes relative to V11**

* TI and TR are read from the headers, not the filenames: the "025" scans are TI = 30.75 ms.
* No `fovshift` (it was defined along the encoding axes, which differ per plane).
* Coil maps are self-calibrated per plane, with one set of coils shared across the four
  TIs and two ENLIVE map sets (section 7). Maps from `T1_graywhite_axi` remain available
  but produce a band artefact.
* The `-p` weight file for unequal spoke counts is now passed as a keyword argument. V11
  prepended it to the argument list, which the BART wrapper places *after* the option
  string, so `-p` would have consumed the next flag as its filename. It never triggered
  in V11 because all TIs had equal counts; here axial TI025 has 992 vs 800.

## 1. Environment

In [ ]:
import os
import re
import ast
import sys
import json
import itertools
from pathlib import Path

import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

BART_PATH = os.environ.get('BART_TOOLBOX_PATH', '/Users/jackewins/bart/')
os.environ['BART_TOOLBOX_PATH'] = BART_PATH
sys.path.append(os.path.join(BART_PATH, 'python'))
from bart import bart, cfl

import ants
import nibabel as nib

%matplotlib inline
plt.rcParams['figure.facecolor'] = 'white'
print('BART toolbox:', BART_PATH, '| ANTsPy', ants.__version__)

## 2. Configuration

In [ ]:
# ---- study -----------------------------------------------------------------
SUBJECT  = '<SUBJECT_ID>'
DATA_DIR = Path('../<SUBJECT_ID>')              # raw .h5, outside the repo
PROC_DIR = Path('../qT1iso_proc') / SUBJECT        # derived data, outside the repo
CFL_DIR  = PROC_DIR / 'CFL'

PLANES   = ('AX', 'COR', 'SAG')
IR_GLOB  = 'T1_map_TI*.user_rrdf_*.h5'
SENS_GLOB = 'T1_graywhite_axi_rrdf_*.h5'
ISO_MM   = 1.8                                     # isotropic target (steps 3-5)

# ---- sensitivity maps ------------------------------------------------------
SENS_METHOD = 'selfcal_shared'   # 'selfcal_shared' | 'selfcal' | 'espirit' | 'enlive'  (section 7)
SENS_NMAPS  = 2                  # ENLIVE map sets for 'selfcal_shared' (2 = soft-SENSE)
RUN_POLE_CHECK   = True          # section 7b: slice-wise phase-pole validation
POLE_CHECK_PLANE = 'COR'
NLINV_CMD  = 'nlinv -m 2 -d 3 -a 30 -b 20 -i 12'   # your optimised ENLIVE values
ECALIB_CMD = 'ecalib -m 1 -r 24:24:12'
SENS_ECHO  = 'matched'   # 'matched': graywhite even->IR even, odd->odd | 'even': even maps for both
FLIP_CHECK = True        # re-derive axis signs from per-coil images (section 5)

# ---- LLR reconstruction ----------------------------------------------------
LLR_LAMBDA = 0.006       # chosen 2026-09-29 from the section 13 sweep (coronal, 2 map sets)
LLR_BLK    = 4
LLR_ITER   = 80          # < 2% change beyond 80 iterations at this lambda
REF_TI_IDX = -1          # longest TI: phase reference / registration target

# ---- lambda x iteration sweep (section 13) ---------------------------------
RUN_SWEEP     = True         # results cached in PROC_DIR/sweep
SWEEP_PLANE   = 'COR'        # R = 6, least TI-to-TI motion
SWEEP_LAMBDAS = [0.005, 0.006, 0.007, 0.008, 0.009, 0.01]   # coarse run: 0.001-0.02 (cached)
SWEEP_ITERS   = [20, 40, 80, 150]

# ---- motion diagnostic -----------------------------------------------------
RUN_MOTION_CHECK = True
SENSE_CMD = 'pics -S -l2 -r 0.01 -i 20 -U'       # per-TI, no coupling across TI

ECHOES = ('even', 'odd')
CFL_DIR.mkdir(parents=True, exist_ok=True)

## 3. rrdf → BART converter

Identical to V11 (`x_coord = arange(n) - n//2`) except: no `fovshift`, and the header
fields needed downstream (geometry string, TI, TR, echo-train duration) go into `info.json`.

In [ ]:
def rrdfconvert(filename, bartname):
    """Convert a Hyperfine rrdf .h5 into BART cfl data + trajectory."""
    with h5py.File(filename, 'r') as f:
        mat  = f['/sequence/matrix_size'][()]
        ksp  = f['data/data_matrix'][()]
        traj = f['data/kspace_coordinate_pe'][()]
        fov  = f['/sequence/fov'][()]
        geom = f['/sequence/acquisition_geometry'][()].decode()
        ti   = float(f['/sequence/ti'][()])
        tr   = float(f['/sequence/tr'][()])
        seq  = ast.literal_eval(f['/sequence/full_sequence_info'][()].decode())
        t_acq = f['/process_info/acquisition_datetime'][()].decode()

    nread, p1, p2 = int(mat[0]), int(mat[1]), int(mat[2])
    x_coord = np.arange(nread) - nread // 2          # dk = 1, DC on-grid

    newtraj = np.swapaxes(traj, 1, 2)
    traj3d  = np.zeros((3, traj.shape[2], traj.shape[1], nread))
    traj3d[0, ...] = np.broadcast_to(
        x_coord, (ksp.shape[2], ksp.shape[3], ksp.shape[1]))
    traj3d[1, ...] = np.transpose(np.broadcast_to(
        p1 * newtraj[:1, ...],
        (ksp.shape[1], 1, ksp.shape[2], ksp.shape[3])), (1, 2, 3, 0))
    traj3d[2, ...] = np.transpose(np.broadcast_to(
        p2 * newtraj[1:, ...],
        (ksp.shape[1], 1, ksp.shape[2], ksp.shape[3])), (1, 2, 3, 0))

    barttraj = np.moveaxis(traj3d[:, :, :, :, None, None, None],
                           [0, 3, 2, 4, 5, 1, 6], [0, 1, 2, 3, 4, 5, 6])
    bartksp  = np.moveaxis(ksp[:, :, :, :, None, None, None],
                           [4, 1, 3, 0, 5, 2, 6], [0, 1, 2, 3, 4, 5, 6])

    cfl.writecfl(f'{bartname}_data', bartksp)
    cfl.writecfl(f'{bartname}_traj', barttraj)

    info = {'fov_mm':   [1000 * float(x) for x in fov],
            'matrix':   [int(x) for x in mat],
            'geom':     geom,
            'TI_s':     ti,
            'TR_s':     tr,
            'etl_s':    float(seq['timing']['etl_duration']),
            'R_header': seq['app_data'].get('undersampling_factor_1.0'),
            'n_pe':     int(ksp.shape[3]),
            'acq_time': t_acq}
    with open(f'{bartname}_info.json', 'w') as fj:
        json.dump(info, fj, indent=2)
    return info


def checked(x):
    """The BART wrapper returns None/0-d on failure; fail loudly instead of caching it."""
    x = np.squeeze(x) if x is not None else None
    if x is None or x.ndim < 3:
        raise RuntimeError('BART call failed - see the error printed above')
    return x.astype(np.complex64)


def convert_one(h5_path, out_stem):
    """Convert if needed, otherwise load the cached info.json."""
    if Path(f'{out_stem}_data.cfl').exists() and Path(f'{out_stem}_info.json').exists():
        return json.load(open(f'{out_stem}_info.json')), 'cached'
    return rrdfconvert(h5_path, out_stem), 'converted'

## 4. Discover and convert the scans

Plane comes from the filename suffix (`_COR`, `_SAG`, none = axial) and is checked
against the header geometry string. TIs are sorted by the **header** TI.

In [ ]:
PLANE_GEOM = {'AX': 'YZX', 'COR': 'XYZ', 'SAG': 'ZXY'}   # (read, pe1, pe2) -> scanner axes
name_re = re.compile(r'T1_map_TI(\d+)_.*?(_COR|_SAG)?\.user_rrdf_(\d{8}_\d{6})\.h5$')

scans = {p: [] for p in PLANES}
for h5 in sorted(DATA_DIR.glob(IR_GLOB)):
    m = name_re.search(h5.name)
    if not m:
        print('  !! skipped (name not recognised):', h5.name); continue
    plane = {'_COR': 'COR', '_SAG': 'SAG', None: 'AX'}[m.group(2)]
    stem  = CFL_DIR / f'{plane}_TI{m.group(1)}_{m.group(3)}'
    info, state = convert_one(h5, str(stem))
    assert info['geom'] == PLANE_GEOM[plane], (h5.name, info['geom'])
    scans[plane].append({'h5': h5.name, 'stem': stem, 'info': info, 'state': state})

for p in PLANES:
    scans[p].sort(key=lambda s: s['info']['TI_s'])

print(f"{'plane':5s} {'TI ms':>7s} {'TR s':>6s} {'geom':>5s} {'matrix':>15s} "
      f"{'n_PE':>5s} {'R':>5s} {'R_hdr':>5s}  acquired         state")
for p in PLANES:
    for s in scans[p]:
        i = s['info']; nx, ny, nz = i['matrix']
        print(f"{p:5s} {1000 * i['TI_s']:7.2f} {i['TR_s']:6.3f} {i['geom']:>5s} "
              f"{str(tuple(i['matrix'])):>15s} {i['n_pe']:5d} {ny * nz / i['n_pe']:5.2f} "
              f"{i['R_header']:5.1f}  {i['acq_time']}  {s['state']}")

TI_s = {p: np.array([s['info']['TI_s'] for s in scans[p]]) for p in PLANES}
TR_s = {p: np.array([s['info']['TR_s'] for s in scans[p]]) for p in PLANES}
for p in PLANES[1:]:
    assert np.allclose(TI_s[p], TI_s[PLANES[0]]), 'TIs differ between planes'

sens_h5 = sorted(DATA_DIR.glob(SENS_GLOB))[-1]
m = re.search(r'(\d{8}_\d{6})\.h5$', sens_h5.name)
sens_stem = CFL_DIR / f'SENS_graywhite_{m.group(1)}'
sens_info, state = convert_one(sens_h5, str(sens_stem))
print(f"\nsens   {sens_h5.name}  geom {sens_info['geom']}  matrix {tuple(sens_info['matrix'])}  "
      f"n_PE {sens_info['n_pe']}  [{state}]")

## 5. Geometry: every scan in one scanner frame

Hyperfine's scanner axes here are **X = S/I, Y = L/R, Z = A/P** (Z along B0). The header
geometry string names the scanner axis of each encoding axis (read, pe1, pe2):

| plane | geometry | slice (pe2, 5 mm) |
|---|---|---|
| axial | `YZX` | X (S/I) |
| coronal | `XYZ` | Z (A/P) |
| sagittal | `ZXY` | Y (L/R) |
| graywhite | `ZYX` | X (S/I) |

All scans share the same auto-align rotation, so in the rotated scanner frame the planes
differ only by an axis permutation and possibly sign flips. Each array becomes an
`ANTsImage` whose direction matrix does the permutation, with voxel `n//2` (BART's image
centre) at the origin.

The permutation is known from the header; the signs are not. `FLIP_CHECK` finds them
by resampling the graywhite **per-coil** images into each plane for all 8 sign
combinations and correlating coil by coil. Coil weighting is asymmetric, so this resolves
an L/R flip that a near-symmetric head would not. Heavily blurred (6 mm) gridding
images are enough for this.

In [ ]:
AXIS = {'X': 0, 'Y': 1, 'Z': 2}


def spacing_of(info):
    return tuple(f / n for f, n in zip(info['fov_mm'], info['matrix']))


def make_image(vol, info, signs=(1, 1, 1)):
    """Wrap a 3D real array on a scan's encoding grid as an ANTsImage in the scanner frame."""
    sp = np.array(spacing_of(info))
    D = np.zeros((3, 3))
    for j, a in enumerate(info['geom']):
        D[AXIS[a], j] = signs[j]
    origin = -D @ (sp * (np.array(vol.shape[:3]) // 2))
    return ants.from_numpy(np.ascontiguousarray(vol, dtype=np.float32),
                           origin=tuple(origin), spacing=tuple(sp), direction=D)


def resample_complex(vol, src_info, dst_info, src_signs, dst_signs):
    """Resample a complex 3D volume from one scan's grid to another's (linear, re/im separately)."""
    dst = make_image(np.zeros(dst_info['matrix'], np.float32), dst_info, dst_signs)
    out = [ants.resample_image_to_target(make_image(c(vol), src_info, src_signs), dst,
                                         interp_type='linear').numpy() for c in (np.real, np.imag)]
    return (out[0] + 1j * out[1]).astype(np.complex64)


def load_echo(stem, e):
    d = cfl.readcfl(f'{stem}_data')[:, :, :, :, :, e]
    t = cfl.readcfl(f'{stem}_traj')[:, :, :, :, :, e]
    return t, d


def grid_coils(stem, info, e=0):
    t, d = load_echo(stem, e)
    return np.squeeze(bart(1, 'nufft -a -d {}:{}:{}'.format(*info['matrix']), t, d))


def ref_scan(p):
    return scans[p][REF_TI_IDX]


SIGNS = {'SENS': (1, 1, 1)}
SIGNS.update({p: (1, 1, 1) for p in PLANES})

if FLIP_CHECK:
    def blurred(stem, info):
        im = np.abs(grid_coils(stem, info))
        sig = [6.0 / s for s in spacing_of(info)]
        return np.stack([gaussian_filter(im[..., c], sig) for c in range(im.shape[-1])], -1)

    gw = blurred(sens_stem, sens_info)
    for p in PLANES:
        s_ref = ref_scan(p)
        pl = blurred(s_ref['stem'], s_ref['info'])
        scores = []
        for sg in itertools.product((1, -1), repeat=3):
            ncc = []
            for c in range(pl.shape[-1]):
                fixed = make_image(pl[..., c], s_ref['info'], sg)
                r = ants.resample_image_to_target(make_image(gw[..., c], sens_info),
                                                  fixed, interp_type='linear').numpy().ravel()
                f, ok = pl[..., c].ravel(), r > 0
                ncc.append(np.corrcoef(f[ok], r[ok])[0, 1])
            scores.append((np.mean(ncc), sg))
        scores.sort(reverse=True)
        SIGNS[p] = scores[0][1]
        print(f"{p:4s} " + '  '.join(f"{''.join('+' if x > 0 else '-' for x in sg)} {v:.3f}"
                                     for v, sg in scores))
print('\nsigns used:', SIGNS)

## 6. Stack the inversion times per plane

As in V11: the four TIs go into BART dim 5, so one `pics` call per plane and echo shares one
data scaling. Axial TI025 has more phase encodes (R = 4.9) than the others (R = 6.1); the
shorter TIs are zero-padded and a `-p` weight file gives the padded samples zero weight.
The different spoke count does not bias anything — that TI simply has slightly less
aliasing and noise.

In [ ]:
def to6(a):
    return a.reshape(a.shape + (1,) * (6 - a.ndim))


def stack_tis(plane, e):
    trajs, datas = zip(*[load_echo(s['stem'], e) for s in scans[plane]])
    tmax = np.max([t.shape for t in trajs], axis=0)
    dmax = np.max([d.shape for d in datas], axis=0)
    padding = any(tuple(t.shape) != tuple(tmax) for t in trajs)
    T, D, P = [], [], []
    for t, d in zip(trajs, datas):
        T.append(np.pad(t, [(0, a - b) for a, b in zip(tmax, t.shape)]))
        D.append(np.pad(d, [(0, a - b) for a, b in zip(dmax, d.shape)]))
        w = np.zeros((1, dmax[1], dmax[2], 1, 1), np.complex64)
        w[0, :d.shape[1], :d.shape[2], 0, 0] = 1.0
        P.append(w)
    T = np.concatenate([to6(x) for x in T], axis=5)
    D = np.concatenate([to6(x) for x in D], axis=5)
    P = np.concatenate([to6(x) for x in P], axis=5) if padding else None
    return T, D, P


for p in PLANES:
    T, D, P = stack_tis(p, 0)
    print(f'{p:4s} traj{T.shape}  data{D.shape}  '
          f'pattern={"none" if P is None else P.shape}')

## 7. Coil sensitivities

`SENS_METHOD` chooses how the maps are made. Every method returns maps shaped
`(X, Y, Z, coil, maps)`; `maps` is 1 except for ENLIVE soft-SENSE.

| value | maps from | notes |
|---|---|---|
| `'selfcal_shared'` *(default)* | each plane's own data, **one set of coils shared across all 4 TIs** (`bart ncalib --shared-col-dims 32`), `SENS_NMAPS` ENLIVE map sets | pure self-calibration; no other scan involved |
| `'selfcal'` | ENLIVE on each plane's reference TI only | R ≈ 6 with no ACS region: noisy, uneven shading |
| `'espirit'` | graywhite, ESPIRiT on Cartesian-gridded k-space, resampled into each plane | band artefact (below) |
| `'enlive'` | graywhite, ENLIVE at its native grid, resampled into each plane | strongest band artefact |

**Why shared across TIs.** Coil sensitivities do not depend on TI, so all four TIs inform
one set of maps while each TI keeps its own image. That is the same data as four
separate calibrations but with four times the samples (and, for axial, different sampling
masks per TI). It is the multi-contrast analogue of the shared-coil model in model-based
IR T1 mapping (Wang et al., MRM 2018;79:730).

**The band artefact and why two map sets (status 2026-09-28: open).** With *one* map set,
every method above, including pure self-calibration, gives a dark band across the
coronal images at the same place. It is not an LLR artefact (it survives removing `-N` and
150 iterations) and not a geometry or FOV-transfer error (pure self-calibration shows it
too; an offset search found none). It disappears with two ENLIVE map sets. Two published
mechanisms predict exactly this:

* NLINV/ENLIVE can converge to a local minimum with a **phase pole** in the image and a
  conjugate pole in the coils; the smoothness penalty forces the coil magnitude to zero
  there, producing a "black hole" in regularised reconstructions. In 3D the poles lie on
  lines (Blumenthal & Uecker, MRM 2026, doi:10.1002/mrm.70333).
* When the data cannot be explained by one set of maps, a relaxed ("soft") SENSE model
  with multiple map sets removes the artefacts from the magnitude image, though the phase
  may still carry the pole (ESPIRiT: Uecker et al., MRM 2014;71:990; ENLIVE: Holme et
  al., Sci Rep 2019;9:3034).

Section 7b tested the phase-pole explanation with the published correction. **It was not
confirmed:** slice-wise calibration shows the band both with and without pole correction.
Also, two map sets are **not a clean fix**: map set 1 carries substantial energy across
the whole head (QC figure below), not just locally, so the 2-map reconstruction has roughly
twice the unknowns and looks softer. The band may be absorbed by that extra freedom rather
than corrected. Two map sets are the default for now because they are artefact-free.
Section 13's lambda sweep shows whether a lower lambda restores sharpness.

**Combining the two map sets.** A soft-SENSE reconstruction returns one image per map set,
`x_m(TI)`. For PSIR every TI must share one phase reference, so each map set is referenced
to its own reference-TI image and the sets are summed:

$$c(TI) = \frac{\sum_m x_m(TI)\,x_m^*(TI_\mathrm{ref})}{\sqrt{\sum_m |x_m(TI_\mathrm{ref})|^2}}$$

At the reference TI this is real, positive and equal to the root-sum-of-squares; at other
TIs its real part carries the polarity. A pole from the maps is identical at every TI
and cancels. With one map set this reduces exactly to V11's PSIR phase correction, which
step 2 therefore no longer needs to repeat.

In [ ]:
def unit_rss(s):
    """Normalise maps (X,Y,Z,coil[,maps]) to unit root-sum-of-squares over coils and maps."""
    s = s if s.ndim == 5 else s[..., None]
    rss = np.sqrt(np.sum(np.abs(s) ** 2, axis=(3, 4), keepdims=True))
    rss = np.where(rss > 1e-9 * rss.max(), rss, np.inf)
    return (s / rss).astype(np.complex64)


def cartesian_grid(t, d, matrix):
    """Nearest-grid-point gridding of non-Cartesian samples; duplicates averaged."""
    tr, dd = np.real(np.squeeze(t)), np.squeeze(d)            # (3, read, enc), (read, enc, coil)
    n = np.array(matrix)
    idx = [np.round(tr[a]).astype(int) + n[a] // 2 for a in range(3)]
    ok = np.all([(idx[a] >= 0) & (idx[a] < n[a]) for a in range(3)], axis=0)
    K = np.zeros(tuple(n) + (dd.shape[-1],), np.complex64)
    C = np.zeros(tuple(n), np.float32)
    loc = tuple(i[ok] for i in idx)
    for c in range(dd.shape[-1]):
        np.add.at(K[..., c], loc, dd[..., c][ok])
    np.add.at(C, loc, 1)
    return K / np.maximum(C, 1)[..., None]


def graywhite_maps(e, i):
    f = PROC_DIR / f'sens_graywhite_{SENS_METHOD}_{e}.npz'
    if f.exists():
        z = np.load(f); return z['s'], z['im'], 'cached'
    t, d = load_echo(sens_stem, i)
    if SENS_METHOD == 'enlive':
        im, s = bart(2, f'{NLINV_CMD} -x {SENS_RESO}', d, t=t)
        im, s = np.squeeze(im), np.squeeze(s)
        s  = s[..., 0] if s.ndim == 5 else s              # first of the two ENLIVE maps
        im = im[..., 0] if im.ndim == 4 else im
    else:
        s = np.squeeze(bart(1, ECALIB_CMD, cartesian_grid(t, d, sens_info['matrix'])))
        s = s[..., 0] if s.ndim == 5 else s
        coil = np.squeeze(bart(1, 'nufft -a -d ' + SENS_RESO, t, d))
        im = np.sum(np.conj(s) * coil, axis=-1)           # coil-combined reference image
    s, im = s.astype(np.complex64), im.astype(np.complex64)
    np.savez(f, s=s, im=im)
    return s, im, 'computed'


def shared_selfcal(p, i, nmaps, extra=''):
    """ENLIVE calibration on all TIs of one plane/echo, coils shared across TI (dim 5)."""
    T, D, P = stack_tis(p, i)
    reso = '{}:{}:{}'.format(*ref_scan(p)['info']['matrix'])
    kw = {'t': T} if P is None else {'t': T, 'p': P}
    s = bart(1, f'ncalib -m {nmaps} -x {reso} --shared-col-dims 32{extra}', D, **kw)
    s = np.squeeze(s) if s is not None else None
    if s is None or s.ndim < 4:
        raise RuntimeError('ncalib failed - see the error printed above')
    return unit_rss(s)


SENS_RESO = '{}:{}:{}'.format(*sens_info['matrix'])
gw_maps, gw_img = {}, {}
sens = {p: {} for p in PLANES}

if SENS_METHOD in ('espirit', 'enlive'):
    for i, e in enumerate(ECHOES):
        gw_maps[e], gw_img[e], state = graywhite_maps(e, i)
        print(f'[{e}] graywhite {SENS_METHOD}: {state}')
    for p in PLANES:
        info = ref_scan(p)['info']
        for e in ECHOES:
            src = gw_maps[e if SENS_ECHO == 'matched' else 'even']
            s = np.stack([resample_complex(src[..., c], sens_info, info, SIGNS['SENS'], SIGNS[p])
                          for c in range(src.shape[3])], axis=-1)
            sens[p][e] = unit_rss(s)
elif SENS_METHOD in ('selfcal', 'selfcal_shared'):
    for p in PLANES:
        s_ref = ref_scan(p)
        reso = '{}:{}:{}'.format(*s_ref['info']['matrix'])
        for i, e in enumerate(ECHOES):
            tag = f'm{SENS_NMAPS}' if SENS_METHOD == 'selfcal_shared' else ''
            f = PROC_DIR / f'sens_{SENS_METHOD}{tag}_{p}_{e}.npz'
            if f.exists():
                sens[p][e] = np.load(f)['s']; continue
            if SENS_METHOD == 'selfcal_shared':
                sens[p][e] = shared_selfcal(p, i, SENS_NMAPS)
            else:
                t, d = load_echo(s_ref['stem'], i)
                _, s = bart(2, f'{NLINV_CMD} -x {reso}', d, t=t)
                s = np.squeeze(s); s = s[..., 0] if s.ndim == 5 else s
                sens[p][e] = unit_rss(s)
            np.savez(f, s=sens[p][e])
else:
    raise ValueError(SENS_METHOD)

SENS_TAG = SENS_METHOD + (f'm{SENS_NMAPS}' if SENS_METHOD == 'selfcal_shared' else '')
for p in PLANES:
    print(f'{p:4s} sens {sens[p]["even"].shape}  ({SENS_TAG})')

In [ ]:
# QC: magnitude of each map set (RSS over coils) and the phase of coil 0 in map set 0,
# mid pe2 slice of each plane. Map set 0 should cover the head; map set 1 (if present)
# should be small except where one set cannot explain the data.
nm = sens[PLANES[0]]['even'].shape[4]
fig, axes = plt.subplots(nm + 1, 3, figsize=(12, 3.6 * (nm + 1)))
for c, p in enumerate(PLANES):
    s = sens[p]['even']; z = s.shape[2] // 2
    sp = spacing_of(ref_scan(p)['info'])
    for m in range(nm):
        axes[m, c].imshow(np.sqrt((np.abs(s[:, :, z, :, m]) ** 2).sum(-1)), cmap='gray',
                          vmin=0, vmax=1, aspect=sp[0] / sp[1])
        axes[m, c].set_title(f'{p}: map set {m}, RSS over coils')
    axes[nm, c].imshow(np.angle(s[:, :, z, 0, 0]), cmap='twilight', vmin=-np.pi, vmax=np.pi,
                       aspect=sp[0] / sp[1])
    axes[nm, c].set_title(f'{p}: phase, coil 0, map set 0')
for a in axes.ravel():
    a.axis('off')
plt.tight_layout(); plt.show()

### 7b. Validation: is the band a phase pole? (slice-wise calibration with pole correction)

Blumenthal & Uecker's phase-pole correction (in BART as `--phase-pole`) is 2D only; in
3D it is refused. Their workaround for 3D data is to inverse-FFT the fully sampled
readout and calibrate slice by slice. Our readout is Cartesian and fully sampled, so the
same works here: each readout position becomes a 2D non-Cartesian problem in the two
phase-encode directions.

This cell calibrates `POLE_CHECK_PLANE` slice by slice with a **single** map set, shared
across TIs, with and without pole correction (12 Gauss-Newton steps, detection after step
8, as in the paper). The correction only acts on slices where a pole is detected.

**Result (2026-09-28, coronal):** the band is present with slice-wise calibration both
with and without pole correction, so a phase pole in the maps does not explain it. Readout
slices are calibrated independently here, so the band, which lies at fixed readout (S/I)
positions, is not produced by 3D map smoothing across slices either. It then runs the same LLR reconstruction
(even echo; fixed at lambda 0.005, 40 iterations, block 4, the settings current when the
check was run) with each and compares them with the 3D maps. If the single-map band
disappears only with pole correction, the phase-pole explanation is confirmed and a
single-map alternative exists.

The per-slice maps have an arbitrary phase per slice. That is harmless here: it is
common to all coils and TIs in a voxel, so it lands in the image phase, and LLR (singular
values) and the PSIR combination are both invariant to it.

In [ ]:
def slicewise_selfcal(p, i, pole):
    """Readout iFFT, then shared-TI ENLIVE calibration per readout slice (single map set)."""
    f = PROC_DIR / f'sens_slicewise_pp{int(pole)}_{p}_{"even" if i == 0 else "odd"}.npz'
    if f.exists():
        return np.load(f)['s']
    T, D, P = stack_tis(p, i)
    nx, ny, nz = ref_scan(p)['info']['matrix']
    Dx = bart(1, 'fft -i -u 2', D)                            # readout k -> x
    if Dx is None or np.shape(Dx) != D.shape:
        raise RuntimeError('readout iFFT failed')
    T2 = T[:, :1].copy(); T2[0] = 0                            # (ky, kz) only
    opt = ' --phase-pole 8' if pole else ''
    maps = np.zeros((nx, ny, nz, D.shape[3], 1), np.complex64)
    for x in range(nx):
        kw = {'t': T2} if P is None else {'t': T2, 'p': P[:, :1]}
        s = bart(1, f'ncalib -m 1 -i 12 -x 1:{ny}:{nz} --shared-col-dims 32{opt}',
                 np.ascontiguousarray(Dx[:, x:x + 1]), **kw)
        maps[x] = unit_rss(np.squeeze(s)[None])[0]
    np.savez(f, s=maps)
    return maps


def map_agreement(a, b):
    """Mean over coils of the correlation of |maps| (map set 0) inside the head support."""
    w = np.sqrt((np.abs(b[..., 0]) ** 2).sum(-1)) > 0.5
    return np.mean([np.corrcoef(np.abs(a[..., c, 0])[w], np.abs(b[..., c, 0])[w])[0, 1]
                    for c in range(a.shape[3])])


if RUN_POLE_CHECK:
    p = POLE_CHECK_PLANE
    T, D, P = stack_tis(p, 0)
    kw = {'t': T} if P is None else {'t': T, 'p': P}
    llr = 'pics -e -S -N -R L:7:7:0.005 -i 40 -b 4 -U'   # fixed: the settings this check was run with
    variants = {'3D, 1 map set': shared_selfcal(p, 0, 1),
                '3D, 2 map sets': sens[p]['even'] if SENS_NMAPS == 2 else shared_selfcal(p, 0, 2),
                'slice-wise, 1 map set': slicewise_selfcal(p, 0, pole=False),
                'slice-wise + pole corr., 1 map set': slicewise_selfcal(p, 0, pole=True)}
    ref3d = variants['3D, 1 map set']
    for k in ('slice-wise, 1 map set', 'slice-wise + pole corr., 1 map set'):
        print(f'|maps| agreement with 3D calibration, {k}: {map_agreement(variants[k], ref3d):.3f}')
    pole_rec = {}
    for k, s in variants.items():
        f = PROC_DIR / f'polecheck_{p}_{k.replace(" ", "").replace(",", "_").replace(".", "")}.npz'
        if f.exists():
            pole_rec[k] = np.load(f)['x']; continue
        x = checked(bart(1, llr, D, s, **kw))
        x = x if x.ndim == 5 else x[..., None, :]
        pole_rec[k] = np.sqrt((np.abs(x) ** 2).sum(3))          # RSS over map sets
        np.savez(f, x=pole_rec[k])
        print(f'{k}: done')

    sp = spacing_of(ref_scan(p)['info'])
    zs = [pole_rec[k].shape[2] // 2 + d for d in (-6, 0, 4)]
    fig, axes = plt.subplots(len(zs), len(variants), figsize=(3.6 * len(variants), 3.8 * len(zs)))
    for c, (k, v) in enumerate(pole_rec.items()):
        for r, z in enumerate(zs):
            sl = v[:, :, z, REF_TI_IDX]
            axes[r, c].imshow(sl, cmap='gray', vmax=np.percentile(sl, 99.5), aspect=sp[0] / sp[1])
            axes[r, c].set_title(f'{k}\nz={z}', fontsize=9); axes[r, c].axis('off')
    plt.suptitle(f'{p}, reference TI, LLR even echo: coil-map variants')
    plt.tight_layout(); plt.show()
else:
    print('RUN_POLE_CHECK = False -- skipped')

## 8. Joint LLR reconstruction per plane

Same regulariser as V11: `-R L:7:7:lambda -b 4`, fully overlapping blocks, FISTA. The trajectory
(`-t`) and weights (`-p`) are passed as wrapper keyword arguments, which the wrapper writes
as `-t <file> -p <file>` — a bare `-t` at the end of the command string would swallow the
`-p` that the wrapper inserts after it. Results are
cached per plane and lambda in `PROC_DIR`; delete the file to force a re-run.

At R ≈ 6 (vs ≈ 1.5 in the phantom data behind V11) the lambda that was right there is
not necessarily right here — section 10 compares against a per-TI SENSE reconstruction
with no coupling across TI, and section 13 sweeps lambda and the iteration count.

In [ ]:
llr_cmd = f'pics -e -S -N -R L:7:7:{LLR_LAMBDA} -i {LLR_ITER} -b {LLR_BLK} -U'
print(llr_cmd, '\n')


def recon_plane(p, cmd, tag):
    f = PROC_DIR / f'recon_{p}_{tag}.npz'
    if f.exists():
        return dict(np.load(f)), 'cached'
    out = {}
    for i, e in enumerate(ECHOES):
        T, D, P = stack_tis(p, i)
        kw = {'t': T} if P is None else {'t': T, 'p': P}
        x = checked(bart(1, cmd, D, sens[p][e], **kw))
        out[e] = x if x.ndim == 5 else x[..., None, :]           # (X, Y, Z, map set, TI)
    np.savez(f, **out)
    return out, 'reconstructed'


def combine_maps(x, ref=REF_TI_IDX):
    """(X,Y,Z,M,TI) -> (X,Y,Z,TI): each map set phase-referenced to its reference-TI image."""
    r = x[..., ref]
    num = np.sum(x * np.conj(r)[..., None], axis=3)
    den = np.sqrt(np.sum(np.abs(r) ** 2, axis=3))[..., None]
    return (num / np.maximum(den, 1e-12 * den.max())).astype(np.complex64)


def rss_maps(x):
    """(X,Y,Z,M,TI) -> (X,Y,Z,TI) magnitude, root-sum-of-squares over map sets."""
    return np.sqrt(np.sum(np.abs(x) ** 2, axis=3)).astype(np.float32)


LLR_TAG = f'{SENS_TAG}_LLR{LLR_LAMBDA}_i{LLR_ITER}_b{LLR_BLK}'
img_maps, img, mag = {}, {}, {}
for p in PLANES:
    img_maps[p], state = recon_plane(p, llr_cmd, LLR_TAG)
    img[p] = {e: combine_maps(img_maps[p][e]) for e in ECHOES}   # phase-referenced, for PSIR
    mag[p] = rss_maps(img_maps[p]['even'])                         # magnitude, for display
    print(f'{p:4s} {state:13s} {img_maps[p]["even"].shape}   max|I| per TI: '
          + '  '.join(f'{mag[p][..., j].max():.3g}' for j in range(len(TI_s[p]))))

## 9. Checkpoint: reconstructed TIs, each plane in its own acquisition plane

Shared window per plane (the longest TI). Each row shows the plane's mid pe2 slice, i.e.
the 1.8 x 1.8 mm in-plane view: axial for AX, coronal for COR, sagittal for SAG. The
near-null TI should be dark, with tissues passing through zero at different TIs.

The second figure shows the real part of the phase-referenced combination (section 7):
signed, so inverted tissue is dark and recovered tissue bright. This is the PSIR input for
step 2, before even/odd echo registration.

In [ ]:
def show_inplane(vols, title, signed=False):
    n = len(TI_s[PLANES[0]])
    fig, axes = plt.subplots(len(PLANES), n, figsize=(3.3 * n, 3.4 * len(PLANES)))
    for r, p in enumerate(PLANES):
        v = vols[p]; z = v.shape[2] // 2
        sp = spacing_of(ref_scan(p)['info'])
        vmax = np.percentile(np.abs(v[:, :, z, REF_TI_IDX]), 99.5)
        for c in range(n):
            sl = np.real(v[:, :, z, c]) if signed else np.abs(v[:, :, z, c])
            axes[r, c].imshow(sl, cmap='gray', vmin=-vmax if signed else 0, vmax=vmax,
                              aspect=sp[0] / sp[1])
            axes[r, c].set_title(f'{p}  TI={1000 * TI_s[p][c]:.0f} ms', fontsize=10)
            axes[r, c].axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.show()


show_inplane(mag, f'LLR even echo (RSS over map sets), {LLR_TAG}')
show_inplane({p: img[p]['even'] for p in PLANES}, 'signed (real part), phase-referenced to TI800',
             signed=True)

## 10. Checkpoint: what does LLR remove? (vs per-TI SENSE)

A lightly Tikhonov-regularised SENSE reconstruction of each TI **independently**
(`SENSE_CMD`) is the reference: it has more noise and aliasing, but no coupling across
TI and no low-rank smoothing. `LLR − SENSE` shows what the regulariser changed. Noise-like
texture and aliasing streaks in the difference are what you want LLR to remove;
anatomical edges (cortex, ventricles, lesions) in the difference mean it is
over-smoothing.

The same SENSE images feed the motion diagnostic in section 11.

In [ ]:
def sense_plane(p):
    f = PROC_DIR / f'sense_{SENS_TAG}_{p}.npz'
    if f.exists():
        return np.load(f)['even']
    out = []
    for s in scans[p]:
        t, d = load_echo(s['stem'], 0)
        x = checked(bart(1, SENSE_CMD, d, sens[p]['even'], t=t))
        out.append(x if x.ndim == 4 else x[..., None])           # (X, Y, Z, map set)
    out = np.stack(out, axis=-1).astype(np.complex64)
    np.savez(f, even=out)
    return out


sense = {p: rss_maps(sense_plane(p)) for p in PLANES}
show_inplane(sense, f'per-TI SENSE, even echo  ({SENSE_CMD})')

In [ ]:
# LLR - SENSE, in-plane mid slice, window 25% of the reference-TI max.
n = len(TI_s[PLANES[0]])
fig, axes = plt.subplots(len(PLANES), n, figsize=(3.3 * n, 3.4 * len(PLANES)))
for r, p in enumerate(PLANES):
    z = sense[p].shape[2] // 2
    sp = spacing_of(ref_scan(p)['info'])
    lim = 0.25 * np.percentile(mag[p][:, :, z, REF_TI_IDX], 99.5)
    for c in range(n):
        dlt = mag[p][:, :, z, c] - sense[p][:, :, z, c]
        axes[r, c].imshow(dlt, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect=sp[0] / sp[1])
        axes[r, c].set_title(f'{p} TI={1000 * TI_s[p][c]:.0f}  |LLR|-|SENSE|', fontsize=9)
        axes[r, c].axis('off')
plt.tight_layout(); plt.show()

## 11. Motion between TIs within each plane

The four TIs of a plane were acquired ~1.5 min apart. Joint LLR assumes a voxel holds the
same tissue at every TI; if the head moved, the block-wise TI matrices stop being low rank
and LLR either leaves artefact or blends neighbouring tissue across TI (a T1 bias at
edges).

Each TI's SENSE magnitude is rigidly registered to the reference TI (mutual information,
since contrast changes with TI). Reported: rotation angle and the mean displacement of
voxels inside the head. Values well under a voxel (1.8 mm in-plane) are fine.
**Near-null TIs are unreliable** — the metric has little structure to lock onto — so read
them with care, and consider a TI flagged only if its neighbours agree.

In [ ]:
def rigid_summary(fwd, fixed_img, mask):
    """Rotation (deg) and mean / max displacement (mm) of masked voxels under a rigid transform."""
    tx = ants.read_transform(fwd[0])
    idx = np.argwhere(mask)[::20]
    pts = np.array([ants.transform_index_to_physical_point(fixed_img, tuple(int(v) for v in i))
                    for i in idx])
    moved = np.array([tx.apply_to_point(tuple(pt)) for pt in pts])
    d = np.linalg.norm(moved - pts, axis=1)
    c = pts.mean(0)                                     # linear part from unit steps
    A = np.stack([np.array(tx.apply_to_point(tuple(c + e))) - tx.apply_to_point(tuple(c))
                  for e in np.eye(3)], axis=1)
    ang = np.degrees(np.arccos(np.clip((np.trace(A) - 1) / 2, -1, 1)))
    return ang, d.mean(), d.max()


motion = {}
if RUN_MOTION_CHECK:
    for p in PLANES:
        info = ref_scan(p)['info']
        mags = sense[p]
        ref = mags[..., REF_TI_IDX]
        fixed = make_image(ref / ref.max(), info, SIGNS[p])
        mask = ref > 0.15 * np.percentile(ref, 99.5)
        motion[p] = []
        for c in range(mags.shape[-1]):
            if c == (REF_TI_IDX % mags.shape[-1]):
                motion[p].append((0.0, 0.0, 0.0)); continue
            mv = make_image(mags[..., c] / mags[..., c].max(), info, SIGNS[p])
            reg = ants.registration(fixed, mv, type_of_transform='Rigid',
                                    aff_metric='mattes', aff_sampling=32)
            motion[p].append(rigid_summary(reg['fwdtransforms'], fixed, mask))
        print(f'{p}')
        for c, (a, dm, dx) in enumerate(motion[p]):
            print(f'   TI={1000 * TI_s[p][c]:6.1f} ms  rot {a:5.2f} deg   '
                  f'mean disp {dm:5.2f} mm   max {dx:5.2f} mm')
else:
    print('RUN_MOTION_CHECK = False -- skipped')

## 12. Checkpoint: do the planes land on each other? (scanner-frame geometry only)

Reference-TI LLR magnitude of each plane resampled onto a common 1.8 mm isotropic grid in
the scanner frame, **no registration**. Red/green/blue = AX/COR/SAG. Grey/white means
agreement; coloured fringes are either head motion between planes (step 4 fixes that) or
a geometry error (a mirror image or a gross offset would be a bug here, not motion).

In [ ]:
ISO_N = int(np.ceil(230 / ISO_MM))
iso_grid = ants.from_numpy(np.zeros((ISO_N,) * 3, np.float32),
                           origin=(-(ISO_N // 2) * ISO_MM,) * 3, spacing=(ISO_MM,) * 3)
iso_ref = {}
for p in PLANES:
    v = mag[p][..., REF_TI_IDX]
    v = v / np.percentile(v, 99.5)
    iso_ref[p] = ants.resample_image_to_target(make_image(v, ref_scan(p)['info'], SIGNS[p]),
                                               iso_grid, interp_type='linear').numpy()

c = ISO_N // 2
views = [('axial (X = S/I fixed)', lambda v: v[c, :, :]),
         ('coronal (Z = A/P fixed)', lambda v: v[:, :, c]),
         ('sagittal (Y = L/R fixed)', lambda v: v[:, c, :])]
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for a, (name, cut) in zip(axes, views):
    rgb = np.stack([cut(iso_ref[p]) for p in PLANES], -1)
    a.imshow(np.clip(rgb, 0, 1)); a.set_title(name); a.axis('off')
plt.suptitle('R = AX, G = COR, B = SAG (reference TI, geometry only)')
plt.tight_layout(); plt.show()

## 13. Lambda × iteration sweep (for visual assessment)

All 12 IR scans share the same acquisition (R ≈ 6; axial TI025 R ≈ 5), so lambda and the
FISTA iteration count are chosen once, on one dataset, and applied everywhere. The two
interact: FISTA starts from zero, so an early-stopped reconstruction is smoother than the
converged one at the same lambda, and a small lambda can look acceptable at 20 iterations
but pick up noise and aliasing by 150.

`SWEEP_PLANE` (default coronal: R = 6 and the least TI-to-TI motion, so motion does not
confound the choice), even echo, same coil maps and block size as section 8. Each
(lambda, iterations) pair is an independent run from zero, **not** a continuation, so each
image is exactly what that setting would produce. Results are cached in
`PROC_DIR/sweep/`; extend `SWEEP_LAMBDAS` / `SWEEP_ITERS` and re-run, and only the new
pairs are computed.

Figures:

1. **Reference TI** (TI800), mid slice: rows = lambda, columns = iterations, plus per-TI
   SENSE (no TI coupling) as the reference in the last column. Shared window.
2. **Near-null TI** (TI150): the TI with the least signal, where the regulariser has the
   most influence and aliasing shows first.
3. **Zoom** on a central patch of the reference TI, where fine structure (sulci, small
   lesions) is visible.
4. **Convergence**: relative change of each image from the longest-iteration run at the
   same lambda. A flat curve means more iterations would change nothing visible.
5. **Line profile** through the zoom patch, reference TI, one line per lambda at the
   longest iteration count: edges that flatten as lambda grows are being smoothed away.

Set `SWEEP_ZOOM` (centre row, column and half-width in voxels) and `SWEEP_SLICE` to look at
a region you care about.

In [ ]:
SWEEP_SLICE = None          # None -> mid pe2 slice
SWEEP_ZOOM  = None          # (row, col, half_width) in voxels; None -> image centre, 24

sweep = {}
if RUN_SWEEP:
    sw_dir = PROC_DIR / 'sweep'; sw_dir.mkdir(exist_ok=True)
    T, D, P = stack_tis(SWEEP_PLANE, 0)
    kw = {'t': T} if P is None else {'t': T, 'p': P}
    for lam in SWEEP_LAMBDAS:
        for it in SWEEP_ITERS:
            f = sw_dir / f'{SWEEP_PLANE}_{SENS_TAG}_L{lam}_i{it}_b{LLR_BLK}.npz'
            if f.exists():
                sweep[lam, it] = np.load(f)['x']; continue
            cmd = f'pics -e -S -N -R L:7:7:{lam} -i {it} -b {LLR_BLK} -U'
            x = checked(bart(1, cmd, D, sens[SWEEP_PLANE]['even'], **kw))
            x = x if x.ndim == 5 else x[..., None, :]
            sweep[lam, it] = x
            np.savez(f, x=x)
            print(f'lambda {lam:<7} iter {it:<4} done')
    print(f'{len(sweep)} sweep reconstructions available')
else:
    print('RUN_SWEEP = False -- skipped')

In [ ]:
if RUN_SWEEP:
    info = ref_scan(SWEEP_PLANE)['info']
    sp = spacing_of(info)
    asp = sp[0] / sp[1]
    zsl = info['matrix'][2] // 2 if SWEEP_SLICE is None else SWEEP_SLICE
    zr, zc, zw = ((info['matrix'][0] // 2, info['matrix'][1] // 2, 24)
                  if SWEEP_ZOOM is None else SWEEP_ZOOM)
    ti_null = int(np.argmin(np.abs(TI_s[SWEEP_PLANE] - 0.15)))
    sw_mag = {k: rss_maps(v) for k, v in sweep.items()}
    ref_sense = sense[SWEEP_PLANE]
    nL, nI = len(SWEEP_LAMBDAS), len(SWEEP_ITERS)

    def grid(ti, crop=None, title=''):
        vmax = np.percentile(sw_mag[SWEEP_LAMBDAS[0], SWEEP_ITERS[-1]][:, :, zsl, REF_TI_IDX], 99.5)
        fig, axes = plt.subplots(nL, nI + 1, figsize=(2.9 * (nI + 1), 2.9 * nL + 0.6))
        for r, lam in enumerate(SWEEP_LAMBDAS):
            for c, it in enumerate(SWEEP_ITERS + ['SENSE']):
                v = (ref_sense if it == 'SENSE' else sw_mag[lam, it])[:, :, zsl, ti]
                if crop is not None:
                    v = v[crop]
                ax = axes[r, c]
                if it == 'SENSE' and r > 0:
                    ax.axis('off'); continue
                ax.imshow(v, cmap='gray', vmin=0, vmax=vmax, aspect=asp,
                          interpolation='nearest')
                ax.set_title('per-TI SENSE' if it == 'SENSE' else f'λ={lam}  it={it}', fontsize=9)
                ax.set_xticks([]); ax.set_yticks([])
        plt.suptitle(title); plt.tight_layout(); plt.show()

    crop = (slice(max(zr - zw, 0), zr + zw), slice(max(zc - zw, 0), zc + zw))
    grid(REF_TI_IDX, None, f'{SWEEP_PLANE} TI={1000 * TI_s[SWEEP_PLANE][REF_TI_IDX]:.0f} ms, slice {zsl}')
    grid(ti_null, None, f'{SWEEP_PLANE} TI={1000 * TI_s[SWEEP_PLANE][ti_null]:.0f} ms (near null), '
                        f'same window as the reference TI')
    grid(REF_TI_IDX, crop, f'zoom: rows {crop[0].start}-{crop[0].stop}, '
                           f'cols {crop[1].start}-{crop[1].stop}')

In [ ]:
if RUN_SWEEP:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for lam in SWEEP_LAMBDAS:
        last = sw_mag[lam, SWEEP_ITERS[-1]]
        ch = [np.linalg.norm(sw_mag[lam, it] - last) / np.linalg.norm(last) for it in SWEEP_ITERS]
        ax[0].plot(SWEEP_ITERS, ch, 'o-', label=f'λ={lam}')
    ax[0].set_xlabel('iterations'); ax[0].set_ylabel(f'rel. change vs {SWEEP_ITERS[-1]} iterations')
    ax[0].set_title('convergence (all TIs, whole volume)'); ax[0].grid(alpha=0.3); ax[0].legend()

    row = zr
    cols = np.arange(max(zc - zw, 0), zc + zw)
    for lam in SWEEP_LAMBDAS:
        ax[1].plot(cols, sw_mag[lam, SWEEP_ITERS[-1]][row, cols, zsl, REF_TI_IDX], label=f'λ={lam}')
    ax[1].plot(cols, sense[SWEEP_PLANE][row, cols, zsl, REF_TI_IDX], 'k:', lw=1, label='SENSE')
    ax[1].set_xlabel('column'); ax[1].set_ylabel('|signal|')
    ax[1].set_title(f'profile row {row}, slice {zsl}, reference TI, {SWEEP_ITERS[-1]} iterations')
    ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

### End of step 1

Things to judge before moving on:

1. **Section 7b**: whether the band is gone with two map sets, and whether slice-wise
   calibration with phase-pole correction removes it with a single map set.
2. **Section 13**: choose `LLR_LAMBDA` and `LLR_ITER` from the sweep, then set them in
   section 2 and re-run sections 8-12 for all planes.
3. **Section 11**: TI-to-TI motion. Motion correction (synthetic-image registration plus
   a motion-compensated reconstruction) is added after lambda and iterations are fixed.
4. **Section 12**: whether the planes overlap to within head motion.